<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 11</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Kafka Streams API (Java)</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Learn how to use the official Kafka Streams API to build a stream processing application.</p>
</div>

---

## 🎯 Overview

In this lab, we'll build a native **Kafka Streams** application using Java and Maven. 

**In this lab, we'll:**
1. Create input and output topics in Kafka
2. Produce test messages to the input topic (using Python)
3. Run our Java Kafka Streams application to transform the messages (convert to uppercase)
4. Consume from the output topic to verify our stream processor

---


## <span style="color: #667eea;">Step 1:</span> Ensure Kafka Cluster is Running

We'll start the basic Kafka environment using the root `docker-compose.yml`.

In [ ]:
!docker-compose -f ../../docker-compose.yml up -d
import time
time.sleep(5)

## <span style="color: #667eea;">Step 2:</span> Create the Input and Output Topics

Let's create two topics for our stream processor.

In [ ]:
!docker exec kafka kafka-topics --bootstrap-server localhost:9092 --create --topic stream-input-topic --partitions 1 --replication-factor 1 --if-not-exists
!docker exec kafka kafka-topics --bootstrap-server localhost:9092 --create --topic stream-output-topic --partitions 1 --replication-factor 1 --if-not-exists
print("\n✓ Topics created successfully!")

## <span style="color: #667eea;">Step 3:</span> Produce Test Messages

Let's write a quick Python script to produce some sample lowercase strings to our `stream-input-topic`.

In [ ]:
from confluent_kafka import Producer

producer = Producer({'bootstrap.servers': 'localhost:9092'})
test_messages = ["hello java streams", "kafka streams api is powerful", "real time processing"]

print("📤 Producing test messages to stream-input-topic...")
for i, msg in enumerate(test_messages):
    producer.produce('stream-input-topic', key=str(i).encode('utf-8'), value=msg.encode('utf-8'))
    print(f"  ✓ Sent: {msg}")
    
producer.flush()
print("\n✅ Done!")

## <span style="color: #667eea;">Step 4:</span> Review the Stream Processor Code

Open `stream-app/src/main/java/com/example/StreamProcessor.java` to see the Java code. The application:
1. Creates a `KStream` from the input topic.
2. Maps the values to UPPERCASE.
3. Writes the transformed stream to the output topic.

### <span style="color: #667eea;">Step 5:</span> Compile Java Streams Application

We will use Maven to compile the Java code

In [ ]:
!sudo apt update
!sudo apt install maven -y

In [ ]:
import os
# Robustly ensure we are in the correct directory in case a previous run crashed
if os.path.basename(os.getcwd()) == 'stream-app':
    os.chdir('..')

print("🔨 Compiling Java project with Maven...")
!mvn -f stream-app/pom.xml clean compile

### <span style="color: #667eea;">Step 6:</span>  Run the Java Streams Application

Run the application in the background.

In [ ]:
import os
import subprocess
import time

print("\n🚀 Starting StreamProcessor in the background...")
# Start the processor in the background
process = subprocess.Popen(
    "mvn exec:java -Dexec.mainClass=com.example.StreamProcessor", 
    cwd="stream-app",
    shell=True
)

print("⏳ Waiting 15 seconds for it to start up and process messages...")
time.sleep(15)

print("🛑 Stopping stream processor...")
process.terminate()
process.wait()

print("✅ Stream processing complete!")

### <span style="color: #667eea;">Step 7:</span> Consume the Transformed Output

Let's consume from `stream-output-topic` to verify that our messages were converted to uppercase by the Java Kafka Streams app!

In [ ]:
print("🔍 Consuming from stream-output-topic:\n")
!docker exec kafka kafka-console-consumer --bootstrap-server localhost:9092 --topic stream-output-topic --from-beginning --max-messages 3 --property print.key=true --property key.separator=" | " --timeout-ms 5000
print("\n✅ Data verified!")

---

<div style="background: linear-gradient(135deg, #38a169 0%, #276749 100%); padding: 30px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white; margin-top: 40px;">
  <h3 style="color: #ffffff; margin-top: 0;">🎉 Lab 11 Complete!</h3>
  <p style="color: #d1fae5; margin: 15px 0;">You've successfully built and run a native Java Kafka Streams application!</p>
</div>
